# Phase 2 – Part B: Structured Prompt Engineering & Generative AI

## 1. Generative AI Model Selection & Setup

We selected OpenAI's gpt-oss-20b open-weight language model and accessed it through GroqCloud using the Groq API.

GroqCloud was selected as the inference provider because it provides API access to the model and supports an OpenAI-compatible interface, which simplifies integration with the Python workflow. The model was selected because it supports structured text generation and can be integrated through GroqCloud. The API key is loaded from environment variables (see Section 2).

The Generative AI component does not perform prediction or diagnosis. Instead, it receives the supervised model output, the probability of the positive screening class, key contributing indicators, and clustering context, then transforms these outputs into concise decision-support briefings.

In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "openai",
    "python-dotenv"
])

print("Required packages installed successfully.")

Required packages installed successfully.


## 2. Environment Setup

This section imports the required libraries and loads the Groq API key securely from the environment variables.

The API key is not stored directly in the notebook to avoid exposing sensitive credentials in the GitHub repository.

In [2]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY was not found in the .env file.")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("Groq client is ready.")

Groq client is ready.


## 3. Prompt Template 1 – Individual Patient Risk Briefing

| Field | Description |
|---|---|
| Template Name & ID | Individual Patient Risk Briefing (PT-01) |
| Intended Use Case | A briefing for one individual patient that explains the supervised model result, the screening probability, the key contributing indicators and the patient's risk segment. |
| Design Rationale | This template combines the supervised prediction, positive-class probability, key contributing indicators, and cluster context into a short individual-level briefing. The structure is designed to preserve the distinction between individual prediction results and group-level cluster characteristics while reducing the risk of unsupported medical conclusions.|
| Prompt Structure | Role, Rules, Patient data, Output format (code cell below). Placeholders: `{prediction}`, `{probability}`, `{top_features}`, `{cluster_segment}`, `{cluster_profile}`. |
| Example Input/Output | See Sections 7 and 8. |
| Assumptions & Limitations | The supervised prediction and cluster assignment are generated before the prompt is called. The probability represents the positive screening class, while the cluster profile describes a group rather than the individual patient. The generated text is for decision support only, may vary between runs, and requires human review. |

**Prompt Design References (apply to both templates)**

1. Groq, "Prompting Guide", https://console.groq.com/docs/prompting
2. OpenAI, "Prompt Engineering", https://developers.openai.com/api/docs/guides/prompt-engineering

Both templates follow these guides: a clear role, rules written as bullet points and placed first, input data separated from instructions with delimiters, and an explicit output format with a length limit.

In [3]:
# Structure based on Groq Prompting Guide and OpenAI Prompt Engineering Guide (references above)
PROMPT_TEMPLATE_1 = """
# Role
You are an assistant for a healthcare screening decision-support system.
You explain the outputs of existing machine learning models for one individual patient.

# Rules
- Do not make an independent medical diagnosis or describe the result as confirmed.
- Use only the information inside <patient_data>. Do not add facts that are not given.
- Do not name specific tests, referrals, medications or symptoms; suggest only general follow-up with a healthcare professional.
- Never use "you" or "your"; always write "the patient".
- Describe the screening result exactly as given; do not rephrase it as an action the patient received.
- State the probability value in the Screening Summary.
- Write the contributing indicators in plain words.
- Distinguish clearly between the individual screening probability and the cluster-level profile.
- Use clear, non-technical language.
- Use the section titles exactly as numbered below, in plain text without bold formatting.
- In "Suggested Follow-up Considerations", do not introduce any new action. Only restate the follow-up implied by the provided screening result in clear language.


# Patient data
<patient_data>
- Screening result: {prediction}
- Probability of a positive screening result: {probability}
- Key contributing indicators: {top_features}
- Risk segment: {cluster_segment}
- Cluster profile: {cluster_profile}
</patient_data>

# Output format
Write a briefing of under 180 words with exactly these sections, and no extra text before or after:
1. Screening Summary
2. Key Contributing Indicators
3. Risk Segment Context
4. Suggested Follow-up Considerations
"""

## 4. Test Case 1 – High-Risk Patient

The first test case represents a patient with a high screening probability and a high-risk clustering profile.

This case is used to evaluate whether the prompt can clearly explain a high-risk screening result without presenting it as a confirmed medical diagnosis.

In [4]:
test_case_1 = {
    "prediction": "Further clinical assessment recommended",
    "probability": "84%",
    "top_features": "HighBP, GenHlth, HighChol",
    "cluster_segment": "High risk",
    "cluster_profile": (
        "Patients with poor general health, many days of poor physical "
        "and mental health, difficulty walking, low physical activity, "
        "lower income, and higher rates of heart disease and stroke."
    )
}

## 5. Test Case 2 – Moderate-Risk Patient

This case represents a patient with a moderate screening probability and a moderate-risk clustering profile. It is used to evaluate whether the prompt balances the screening result with a less severe segment profile.

In [5]:
test_case_2 = {
    "prediction": "Further clinical assessment recommended",
    "probability": "61%",
    "top_features": "HighBP, GenHlth, HighChol",
    "cluster_segment": "Moderate risk",
    "cluster_profile": (
        "Older patients with high blood pressure and high cholesterol who "
        "still report good physical and mental health and few mobility problems."
    )
}

## 6. Test Case 3 – Low-Risk Patient

This case represents a patient with a low screening probability and a low-risk clustering profile. It is used to evaluate whether the prompt avoids overstating risk when the screening result is routine.

In [6]:
test_case_3 = {
    "prediction": "Routine follow-up",
    "probability": "24%",
    "top_features": "HighBP, GenHlth, HighChol",
    "cluster_segment": "Low risk",
    "cluster_profile": (
        "Younger patients with mostly normal blood pressure and cholesterol, "
        "good general health, physically active and with higher income."
    )
}

## 7. Constructing the Prompt for Test Case 1

The values from Test Case 1 are inserted into the placeholders defined in Prompt Template 1.

This creates the final prompt that will be sent to the Generative AI model.

In [7]:
prompt_case_1 = PROMPT_TEMPLATE_1.format(**test_case_1)

print(prompt_case_1)


# Role
You are an assistant for a healthcare screening decision-support system.
You explain the outputs of existing machine learning models for one individual patient.

# Rules
- Do not make an independent medical diagnosis or describe the result as confirmed.
- Use only the information inside <patient_data>. Do not add facts that are not given.
- Do not name specific tests, referrals, medications or symptoms; suggest only general follow-up with a healthcare professional.
- Never use "you" or "your"; always write "the patient".
- Describe the screening result exactly as given; do not rephrase it as an action the patient received.
- State the probability value in the Screening Summary.
- Write the contributing indicators in plain words.
- Distinguish clearly between the individual screening probability and the cluster-level profile.
- Use clear, non-technical language.
- Use the section titles exactly as numbered below, in plain text without bold formatting.
- In "Suggested Follow-up C

## 8. Generative AI API Call

The completed prompt is sent to the selected Generative AI model through the API.

The model generates a short explanation of the screening result for one patient, based only on the supplied machine learning outputs and clustering information.

In [8]:
response = client.responses.create(
    model="openai/gpt-oss-20b",
    input=prompt_case_1,
    temperature=0.2
)

template1_case1_output = response.output_text

print(template1_case1_output)

1. Screening Summary  
The patient has a high probability (84%) of a positive screening result, and the model recommends further clinical assessment.  

2. Key Contributing Indicators  
The main factors are high blood pressure, general health concerns, and high cholesterol.  

3. Risk Segment Context  
The patient falls into a high risk group. The cluster includes individuals with poor general health, many days of poor physical and mental health, difficulty walking, low physical activity, lower income, and higher rates of heart disease and stroke.  

4. Suggested Follow-up Considerations  
The patient should seek additional clinical evaluation to address these concerns.


## 9. Generating and Saving the Template 1 Outputs

All three test cases are inserted into Prompt Template 1, sent to the model, and saved as text files so that the outputs can be reviewed later and compared with the outputs produced by the second prompt template.

Saving the outputs also allows the project results to be reviewed without requiring the API to be called again.

In [9]:
from pathlib import Path

output_dir = Path("example_outputs")
output_dir.mkdir(exist_ok=True)

test_cases = [test_case_1, test_case_2, test_case_3]
template1_outputs = {}

for i, case in enumerate(test_cases, start=1):
    prompt = PROMPT_TEMPLATE_1.format(**case)

    response = client.responses.create(
        model="openai/gpt-oss-20b",
        input=prompt,
        temperature=0.2
    )
    template1_outputs[i] = response.output_text

    with open(output_dir / f"template1_case{i}.txt", "w", encoding="utf-8") as f:
        f.write(template1_outputs[i])

    print(f"--- Case {i} ({case['cluster_segment']}) ---")
    print(template1_outputs[i], "\n")

print("All Template 1 cases saved successfully.")

--- Case 1 (High risk) ---
1. Screening Summary  
The patient has a screening result indicating that further clinical assessment is recommended. The probability of a positive screening result is 84%.

2. Key Contributing Indicators  
The patient’s high blood pressure, general health concerns, and high cholesterol levels contributed to the screening outcome.

3. Risk Segment Context  
The patient belongs to a high risk segment. The cluster profile describes individuals with poor general health, many days of poor physical and mental health, difficulty walking, low physical activity, lower income, and higher rates of heart disease and stroke.

4. Suggested Follow-up Considerations  
The screening result recommends that the patient undergo further clinical assessment. 

--- Case 2 (Moderate risk) ---
1. Screening Summary  
The screening indicates that further clinical assessment is recommended. The probability of a positive screening result is 61%.

2. Key Contributing Indicators  
High bl

## 10. Prompt Template 2 – Individual Executive Risk Summary

| Field | Description |
|---|---|
| Template Name & ID | Individual Executive Risk Summary (PT-02) |
| Intended Use Case | A short executive summary of the expected risk of one individual patient, written for a decision-maker and focused on the risk level, the key drivers and the next steps. |
| Design Rationale | This template uses the same patient-level inputs as Template 1 but changes the communication objective. Instead of primarily explaining the model outputs, it emphasizes decision-oriented information by adding segment size, segment diabetes rate, and concise suggested next steps for an executive audience. Using the same cases enables a fair comparison between the two prompt strategies. |
| Prompt Structure | Role, Rules, Patient data, Output format (code cell below). Placeholders: `{prediction}`, `{probability}`, `{top_features}`, `{cluster_segment}`, `{size_pct}`, `{diabetic_pct}`, `{cluster_profile}`. |
| Example Input/Output | See Sections 14 and 15. |
| Assumptions & Limitations | The probability is the chance of a positive screening result. Segment percentages describe this dataset only and describe the group, not the patient. Outputs vary slightly between runs even at temperature 0.2. Output must be reviewed by a person. |

In short: Template 1 is an individual explanatory briefing, and Template 2 is an executive decision-oriented summary.

Template 2 follows the same prompt design references as Template 1 (see Section 3).


In [10]:
# Structure based on Groq Prompting Guide and OpenAI Prompt Engineering Guide (see references in Section 3)
PROMPT_TEMPLATE_2 = """
# Role
You are an assistant for a healthcare screening decision-support system.
You write short executive risk summaries for decision-makers, based on the outputs of existing machine learning models for one individual patient.

# Rules
- Do not make any medical diagnosis, and do not present suggestions as medical advice or confirmed findings.
- Use only the information inside <patient_data>. Do not add facts, interpretations or conditions that are not given (for example "burden of disease" or "comorbid conditions"), and do not compute new figures.
- Describe the screening result exactly as given; do not say that anything has been scheduled, ordered or done.
- Do not explain causes or links between factors unless they are given in the data.
- Suggested next steps must remain general and must be supported by the provided screening result (for example, follow-up with a healthcare professional). Do not name specific tests, screenings, treatments, medications, specialists, symptoms, referrals, lifestyle interventions or monitoring actions unless they are explicitly given in the input.
- Never use "you" or "your"; always write "the patient".
- State the probability exactly as given. In "Expected Risk Level", state that the segment diabetes rate describes the group, not the patient.
- Write the key drivers in plain words, covering both the key indicators and the main characteristics of the cluster profile.
- Use clear, non-technical language suitable for a non-clinical decision-maker.
- Use the section titles exactly as numbered below, in plain text without bold formatting.
- Write the "Suggested Next Steps" section as bullet points.
- Keep the total under 180 words.
- Distinguish clearly between the individual screening probability and the cluster-level profile; the cluster profile describes the group, not the patient.

# Patient data
<patient_data>
- Screening result: {prediction}
- Probability of a positive screening result: {probability}
- Key contributing indicators: {top_features}
- Risk segment: {cluster_segment}
- Segment size: {size_pct}% of all patients
- Diabetes rate inside this segment: {diabetic_pct}% of the patients in this segment have diabetes
- Cluster profile: {cluster_profile}
</patient_data>

# Output format
Write a briefing of under 180 words with exactly these sections, and no extra text before or after:
1. Executive Summary
2. Expected Risk Level
3. Key Drivers
4. Suggested Next Steps
"""

## 11. Test Case 1 – High-Risk Patient

This case uses the same patient as Test Case 1 of Template 1, with the size and diabetes rate of the patient's segment added. Both templates are therefore compared on identical inputs.

In [11]:
t2_case_1 = {**test_case_1, "size_pct": 23.3, "diabetic_pct": 75.8}

## 12. Test Case 2 – Moderate-Risk Patient

This case uses the same patient as Test Case 2 of Template 1, with the size and diabetes rate of the patient's segment added.

In [12]:
t2_case_2 = {**test_case_2, "size_pct": 40.9, "diabetic_pct": 63.4}

## 13. Test Case 3 – Low-Risk Patient

This case uses the same patient as Test Case 3 of Template 1, with the size and diabetes rate of the patient's segment added.

In [13]:
t2_case_3 = {**test_case_3, "size_pct": 35.8, "diabetic_pct": 20.3}

t2_cases = [t2_case_1, t2_case_2, t2_case_3]

## 14. Constructing the Prompt for Test Case 1

The values from Test Case 1 are inserted into the placeholders defined in Prompt Template 2. This creates the final prompt that is sent to the Generative AI model.

In [14]:
prompt_t2_case_1 = PROMPT_TEMPLATE_2.format(**t2_case_1)

print(prompt_t2_case_1)


# Role
You are an assistant for a healthcare screening decision-support system.
You write short executive risk summaries for decision-makers, based on the outputs of existing machine learning models for one individual patient.

# Rules
- Do not make any medical diagnosis, and do not present suggestions as medical advice or confirmed findings.
- Use only the information inside <patient_data>. Do not add facts, interpretations or conditions that are not given (for example "burden of disease" or "comorbid conditions"), and do not compute new figures.
- Describe the screening result exactly as given; do not say that anything has been scheduled, ordered or done.
- Do not explain causes or links between factors unless they are given in the data.
- Suggested next steps must remain general and must be supported by the provided screening result (for example, follow-up with a healthcare professional). Do not name specific tests, screenings, treatments, medications, specialists, symptoms, referr

## 15. Generative AI API Call

The completed prompt is sent to the selected Generative AI model through the API. This example shows the summary generated for the High-risk patient.

In [15]:
response = client.responses.create(
    model="openai/gpt-oss-20b",
    input=prompt_t2_case_1,
    temperature=0.2
)

template2_case1_output = response.output_text

print(template2_case1_output)

1. Executive Summary  
The patient has an 84% probability of a positive screening result. The screening indicates that further clinical assessment is recommended.

2. Expected Risk Level  
The patient is classified in a high risk segment that comprises 23.3% of all patients. Within this segment, 75.8% of patients have diabetes. This diabetes rate describes the group, not the patient.

3. Key Drivers  
The screening highlights high blood pressure, general health status, and high cholesterol as key contributing indicators. The cluster profile shows patients with poor general health, many days of poor physical and mental health, difficulty walking, low physical activity, lower income, and higher rates of heart disease and stroke.

4. Suggested Next Steps  
- The patient should follow up with a healthcare professional for further assessment.  
- The patient may discuss the high probability of a positive screening result and associated risk factors with a clinician.


## 16. Generating and Saving the Template 2 Outputs

Each test case is inserted into Prompt Template 2 and sent to the model. The responses are saved as text files so they can be compared with the Template 1 outputs without calling the API again.

In [16]:
template2_outputs = {}

for i, case in enumerate(t2_cases, start=1):
    prompt = PROMPT_TEMPLATE_2.format(**case)

    response = client.responses.create(
        model="openai/gpt-oss-20b",
        input=prompt,
        temperature=0.2
    )
    template2_outputs[i] = response.output_text

    with open(output_dir / f"template2_case{i}.txt", "w", encoding="utf-8") as f:
        f.write(template2_outputs[i])

    print(f"--- Case {i} ({case['cluster_segment']}) ---")
    print(template2_outputs[i], "\n")

--- Case 1 (High risk) ---
1. Executive Summary  
The screening result recommends further clinical assessment. The probability of a positive screening result is 84%.

2. Expected Risk Level  
The patient is categorized in a high‑risk segment, which comprises 23.3% of all patients. Within this segment, 75.8% of patients have diabetes. This statistic describes the group, not the patient’s individual status.

3. Key Drivers  
Key contributing indicators are high blood pressure, general health concerns, and high cholesterol. The cluster profile includes poor general health, many days of poor physical and mental health, difficulty walking, low physical activity, lower income, and higher rates of heart disease and stroke.

4. Suggested Next Steps  
- The patient should be referred for further clinical assessment.  
- The patient should discuss these findings with a healthcare professional.  
- The patient should consider lifestyle factors that may influence blood pressure, cholesterol, and g

## 17. Testing Framework – Quantitative Metrics

Each Template 1 output is compared with the Template 2 output of the same risk segment (High, Moderate and Low). Four quantitative metrics are computed: response length, keyword alignment with the segment profile, and two readability scores (Flesch-Kincaid grade and Flesch reading ease).

In [17]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "textstat", "pandas"])

import re
import json
import pandas as pd
import textstat

# Each pair matches the Template 1 case and the Template 2 case of the same risk segment
pairs = [
    {"segment": "High risk",     "t1_case": 1, "t2_case": 1},
    {"segment": "Moderate risk", "t1_case": 2, "t2_case": 2},
    {"segment": "Low risk",      "t1_case": 3, "t2_case": 3},
]

# Domain keywords that a good briefing should mention for each segment
keywords = {
    "High risk":     ["poor", "walking", "physical activity", "income", "heart disease", "stroke"],
    "Moderate risk": ["older", "blood pressure", "cholesterol", "good", "mobility"],
    "Low risk":      ["younger", "normal", "blood pressure", "cholesterol", "active", "income"],
}


def read_output(template, case_number):
    path = output_dir / f"template{template}_case{case_number}.txt"
    return path.read_text(encoding="utf-8")


def keyword_alignment(text, words):
    text = text.lower()
    return round(sum(word in text for word in words) / len(words), 2)


rows = []
for pair in pairs:
    for template, case_number in ((1, pair["t1_case"]), (2, pair["t2_case"])):
        text = read_output(template, case_number)
        rows.append({
            "Segment": pair["segment"],
            "Template": f"Template {template}",
            "Words": len(text.split()),
            "Keyword alignment": keyword_alignment(text, keywords[pair["segment"]]),
            "Flesch-Kincaid grade": textstat.flesch_kincaid_grade(text),
            "Reading ease": textstat.flesch_reading_ease(text),
        })

metrics_df = pd.DataFrame(rows)
metrics_df

,Segment,Template,Words,Keyword alignment,Flesch-Kincaid grade,Reading ease
0,High risk,Template 1,105,1.0,13.823571,27.123929
1,High risk,Template 2,142,1.0,10.405198,41.093744
2,Moderate risk,Template 1,87,1.0,12.986839,27.779569
3,Moderate risk,Template 2,133,1.0,9.986783,44.511853
4,Low risk,Template 1,81,1.0,12.943273,29.694909
5,Low risk,Template 2,145,1.0,9.633451,47.072054


## 18. Testing Framework – Simulated User Preference

A second language model, `qwen/qwen3.8-27b` (served through the same Groq API), is asked to act as a reviewer and rate each briefing from 1 (poor) to 5 (excellent) on the five qualitative criteria, using the original input data as the reference for factual accuracy. The briefings were written by `openai/gpt-oss-20b`, so the reviewer is a different model. This is a simulation, not a real user study, so the scores are an indication only.

In [18]:
CRITERIA = [
    "Relevance",
    "Detail & Completeness",
    "Clarity & Readability",
    "Personalization",
    "Safety & Factual Accuracy",
]

JUDGE_PROMPT = """
You are a reviewer of healthcare decision-support briefings.

Rate the briefing from 1 (poor) to 5 (excellent) on each criterion:
- Relevance: does it match the source data and the recommended action?
- Detail & Completeness: does it give enough information?
- Clarity & Readability: can a non-expert understand it?
- Personalization: does it reflect the patient or segment context?
- Safety & Factual Accuracy: does it avoid facts that are not in the source data?

Source data:
{source}

Briefing:
{briefing}

Return only a JSON object whose keys are exactly: {criteria}. Each value must be an integer from 1 to 5.
"""


import time
from openai import RateLimitError

JUDGE_MODEL = "qwen/qwen3.8-27b"


def judge(briefing, source):
    prompt = JUDGE_PROMPT.format(
        source=json.dumps(source, indent=2),
        briefing=briefing,
        criteria=", ".join(f'"{c}"' for c in CRITERIA),
    )
    for _ in range(3):
        text = ""
        for attempt in range(5):
            try:
                response = client.chat.completions.create(
                    model=JUDGE_MODEL,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0,
                )
                text = response.choices[0].message.content or ""
                break
            except RateLimitError:
                time.sleep(30 * (attempt + 1))
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
        for found in reversed(re.findall(r"\{[^{}]*\}", text, re.S)):
            try:
                scores = json.loads(found)
            except json.JSONDecodeError:
                continue
            if all(c in scores for c in CRITERIA):
                return {c: int(scores[c]) for c in CRITERIA}
    raise ValueError("The reviewer did not return valid JSON.")


score_rows = []
for pair in pairs:
    sources = {1: test_cases[pair["t1_case"] - 1], 2: t2_cases[pair["t2_case"] - 1]}
    cases = {1: pair["t1_case"], 2: pair["t2_case"]}
    for template in (1, 2):
        scores = judge(read_output(template, cases[template]), sources[template])
        score_rows.append({"Segment": pair["segment"], "Template": f"Template {template}", **scores})

scores_df = pd.DataFrame(score_rows)
scores_df

,Segment,Template,Relevance,Detail & Completeness,Clarity & Readability,Personalization,Safety & Factual Accuracy
0,High risk,Template 1,5,4,5,4,5
1,High risk,Template 2,5,5,5,4,5
2,Moderate risk,Template 1,5,4,5,4,5
3,Moderate risk,Template 2,5,4,5,4,5
4,Low risk,Template 1,5,4,5,4,5
5,Low risk,Template 2,5,4,5,4,5


## 19. Summary of Results

The table below shows the average of each metric for each template across the three test cases. The summary is also saved as a CSV file.

In [19]:
summary_df = pd.concat([
    metrics_df.groupby("Template")[["Words", "Keyword alignment", "Flesch-Kincaid grade", "Reading ease"]].mean(),
    scores_df.groupby("Template")[CRITERIA].mean(),
], axis=1).round(2)

summary_df.to_csv(output_dir / "comparison_summary.csv")
summary_df.T

Template,Template 1,Template 2
Words,91.00,140.00
Keyword alignment,1.00,1.00
Flesch-Kincaid grade,13.25,10.01
Reading ease,28.20,44.23
Relevance,5.00,5.00
Detail & Completeness,4.00,4.33
Clarity & Readability,5.00,5.00
Personalization,4.00,4.00
Safety & Factual Accuracy,5.00,5.00


## 20. Analysis of Results

Both templates were tested on the same three patients (High, Moderate and Low risk). The analysis combines a manual reading of every output with the metrics in Sections 17–19. With only three cases and a simulated reviewer, the results are indicative, not conclusive.

### Quantitative results

| Metric | Template 1 | Template 2 |
|---|---|---|
| Response length (words) | 91.0 | 140.0 |
| Keyword alignment | 1.00 | 1.00 |
| Flesch-Kincaid grade (lower is easier) | 13.25 | 10.01 |
| Flesch reading ease (higher is easier) | 28.20 | 44.23 |
| Relevance (1–5) | 5.00 | 5.00 |
| Detail & Completeness (1–5) | 4.00 | 4.33 |
| Clarity & Readability (1–5) | 5.00 | 5.00 |
| Personalization (1–5) | 4.00 | 4.00 |
| Safety & Factual Accuracy (1–5) | 5.00 | 5.00 |

- **Length:** both templates stay under the 180-word limit. Template 2 is about 49 words longer because it adds the segment size, the diabetes rate and a list of next steps.
- **Keyword alignment:** both templates mention the cluster profile in all three cases in this run. In an earlier run Template 2 scored 0.39 because its Key Drivers section left out the cluster profile, so this metric changes between runs.
- **Readability:** Template 2 is easier to read (grade 10.0 against 13.3), but both remain difficult texts. Part of the score comes from long medical terms.

### Qualitative observations (manual reading)

- **Relevance:** both templates report the screening result and the probability correctly in all three cases.
- **Detail & Completeness:** Template 2 adds the segment size, the diabetes rate and a list of next steps. Template 1 is shorter and keeps to the individual result.
- **Clarity & Readability:** Template 2 is easier to scan, but it is longer and carries more numbers. In the Low-risk case Template 1 added a dash before each line, a small deviation from the requested format.
- **Personalization:** both templates use the same indicators and the cluster profile. In the Moderate-risk case Template 2 wrote "the patient's older age", which presents a group trait as the patient's own, although age was not one of the given indicators.
- **Indicators read as the patient's conditions (both templates):** the same global indicators (HighBP, GenHlth, HighChol) are given for every patient. Template 1 wrote "The patient's high blood pressure, general health concerns, and high cholesterol levels contributed to the screening outcome" (High risk), and both templates listed high blood pressure and high cholesterol for the Low-risk patient, whose cluster profile describes normal blood pressure and cholesterol.
- **Safety & Factual Accuracy (Template 1):** in all three cases it stated the probability as given, described the cluster profile as a group characteristic and did not say that anything had been done. Its follow-up section only restated the follow-up implied by the screening result ("further clinical assessment" or "routine follow-up"), without lifestyle, monitoring or other new actions. In earlier runs, before the follow-up rule was added to the prompt, this section contained "potential interventions", "assessment of cardiovascular health", "review of lifestyle habits, diet, and physical activity", "monitoring", and once "possible referrals for cardiovascular assessment".
- **Safety & Factual Accuracy (Template 2):** it broke its own rules in all three cases: "The patient should be referred for further clinical assessment" and lifestyle advice (High risk); "Review the patient's blood pressure and cholesterol management plan" and "Consider routine monitoring of blood glucose and other relevant health markers" (Moderate risk); "The patient is scheduled for a routine follow-up", "continue routine monitoring as scheduled", "maintain healthy lifestyle habits" and "discuss any new symptoms" (Low risk). The diabetes rate was marked as a group figure in all three cases. In earlier runs it also wrote "Arrange a follow-up appointment", "will test positive for the condition" and "in the database".

### Reviewer scores compared with manual reading
A second model (`qwen/qwen3.8-27b`) rated the briefings written by `openai/gpt-oss-20b`. It gave both templates the same scores for Relevance, Clarity, Personalization and Safety, and a slightly higher Detail score to Template 2 (4.33 against 4.00). It did not detect the rule violations that the manual reading found in Template 2, nor the indicators that read as the patient's conditions. In an earlier run, the writing model acting as reviewer also gave both templates a Safety score of 5.00. The reviewers therefore did not separate the templates, and the manual reading has more weight, especially for safety.

### Limits of this analysis
- Three test cases only, one final run per case. The output varies between runs even at temperature 0.2: between two earlier runs of the identical Template 2 prompt and inputs, keyword alignment changed from 0.39 to 1.00 and the average length from 122 to 146 words.
- Between the last two runs, the follow-up rule of Template 1 and the indicators of the test cases were changed together. With one run per case, we cannot tell how much of the improvement of Template 1 is due to the new rule and how much to natural variation.
- The reviewer is a language model, not a person, and its scores did not separate the templates.
- No real users were involved, so "user preference" is simulated.
- The keyword check is lexical and does not detect wrong or misleading statements.

## 21. Best Prompt Selection & Justification

Template 1 – Individual Patient Risk Briefing (PT-01) was selected as the prompt template for the final decision-support system.

### Evidence

- **Rule compliance (manual reading):** in the final run Template 1 stated the probability as given, described the cluster profile as a group characteristic, did not say that anything had been done, and only restated the follow-up implied by the screening result in all three cases. Template 2 broke its own rules in all three cases: it suggested a referral, monitoring of blood glucose and a management plan, said that the follow-up was "scheduled", added lifestyle and symptom advice, and presented a group trait (older age) as the patient's own.
- **Shared weakness:** with the same global indicators for every patient, both templates sometimes wrote them as the patient's own conditions, including for the Low-risk patient.
- **Keyword alignment:** 1.00 for both templates in this run. An earlier run gave 0.39 for Template 2, so this metric did not separate the templates reliably.
- **Length:** about 91 words for Template 1 and 140 words for Template 2.
- **Readability:** Template 2 is easier to read (Flesch-Kincaid grade 10.01 against 13.25, reading ease 44.23 against 28.20).
- **Simulated reviewer:** both templates scored 5.00 for Relevance, Clarity and Safety & Factual Accuracy, and 4.00 for Personalization. Template 2 scored slightly higher for Detail & Completeness (4.33 against 4.00). The reviewer did not detect the rule violations of Template 2, so the manual reading was given more weight.

### Alignment with system goals

Because the system supports healthcare screening decisions, three things were prioritized over readability and extra detail: a clear separation between the individual prediction and the cluster-level information, no statements that are not supported by the input, and consistent use of the given inputs. Template 1 met these more consistently in our tests. Template 2 remains an interesting executive-summary format that is easier to read and more detailed, but its extra detail came with more unsupported statements.

Ethical considerations and the wider limitations of the system are discussed in Section 23.

### Limitations of this selection

- Only three test cases and one final run per case. The outputs vary between runs, and the follow-up rule of Template 1 and the test indicators changed together between the last two runs, so the improvement of Template 1 cannot be attributed to the rule alone.
- Template 1 is not free of errors either: its indicators can read as the patient's own conditions, and in one case it added dashes that were not requested.
- The reviewer is a language model, not a person, and it did not separate the templates. No real users or clinicians took part.
- Template 1 is still written at a college reading level.

### Possible improvements

- Labelling the indicator field in the prompt as "most influential features of the model (the same for all patients)" and adding a rule not to describe them as the patient's own conditions (not tested here).
- Shorter sentences and simpler wording in Template 1 to lower the reading grade.
- Testing a version of Template 1 that also includes the segment size and diabetes rate, with an explicit note that they describe the group.
- For Template 2, allowing one to three next-step bullets instead of a required list, since the required list may push the model to add unsupported steps (a hypothesis not tested here).
- Running each case several times and counting rule violations automatically.
- Using several reviewer models with stricter instructions (for example asking them to list every unsupported statement), and a user study with clinicians or decision-makers.
- Adding an automatic check for forbidden words (for example "scheduled" or "referred") before an output is shown to the user.

## 22. Integration Plan for the Final System

The selected prompt (Template 1 – Individual Patient Risk Briefing, PT-01) is the final explanation layer of the healthcare screening decision-support pipeline. It does not replace either machine learning model. Its purpose is to turn their outputs into a concise and understandable briefing for decision support.

### Integration steps

1. **Input:** the user enters the required patient health indicators in the final interface.
2. **Supervised prediction:** the supervised model produces the probability of the positive screening class. This probability is converted into a screening result using the decision threshold selected in Phase 1 (0.35): "Further clinical assessment recommended" at or above the threshold, and "Routine follow-up" below it. The main contributing indicators are the top features of the supervised model (global feature importance).
3. **Cluster assignment:** the clustering component assigns the patient to one of the three risk segments (Low, Moderate or High risk) and provides the cluster profile of that segment.
4. **Prompt construction:** these outputs are inserted into the placeholders of Template 1: `{prediction}`, `{probability}`, `{top_features}`, `{cluster_segment}` and `{cluster_profile}`.
5. **Generation:** the completed prompt is sent to `openai/gpt-oss-20b` through the Groq API, with the same setting used in testing (temperature 0.2).
6. **Display:** the generated briefing is shown on the dashboard together with the original prediction, probability, risk segment and visual analytics.

The final pipeline can be summarized as:

Patient Health Indicators → Supervised Prediction → Cluster Assignment → Generative AI Briefing → Executive Dashboard

### Safeguards

- The Groq API key is read from an environment variable (or from Streamlit secrets when deployed) and is never stored in the code or in the repository.
- The prompt used in the final pipeline is identical to Template 1 in this notebook, so the tested behavior applies to the final system.
- The prediction, probability and risk segment shown on the dashboard come directly from the machine learning models, not from the generated text.
- The dashboard states that the tool is a screening aid and not a medical diagnosis, and the briefing is intended to be reviewed by a person.
- If the API call fails or no API key is available, the dashboard shows a fixed message that the AI summary is not available, and the model outputs are still displayed.
- As a possible improvement (see Section 21), an automatic check for forbidden words can be applied to the generated text before it is shown.

## 23. Ethical Considerations & Limitations

### Ethical considerations

- **Decision support, not diagnosis:** the Generative AI component only rephrases the outputs of the two machine learning models. Its prompt forbids diagnoses and confirmed findings, the dashboard states that the tool is a screening aid, and every briefing is intended to be reviewed by a person before any decision is made.
- **Unsupported statements:** language models can add information that is not in the input. This happened in our tests. Template 2 suggested a referral, monitoring and lifestyle advice and said that the follow-up was already scheduled, although its rules forbid this. Template 1 added lifestyle, monitoring and assessment suggestions in earlier runs, before a stricter follow-up rule was added to its prompt. For this reason the prompt contains explicit rules, the model outputs (probability, risk segment) are displayed separately from the generated text, and an automatic check of the generated text is proposed as a future safeguard.
- **Individual versus group information:** the cluster profile describes a group, not the patient. Presenting group traits (for example lower income) as personal facts could be inaccurate and stigmatizing. The prompt therefore asks the model to keep the individual prediction and the cluster profile clearly separate.
- **Over-reliance:** fluent text can look more certain than it is. The probability is an estimate from a supervised model and should not be read as a certain outcome. The screening threshold (0.35) was chosen to catch more at-risk people, which means that some patients will be flagged who are not at risk, and the wording should not create unnecessary alarm.
- **Privacy:** the prompt sent to the external API contains no names or identifiers and does not include the patient's raw answers. It contains only the screening result, the probability, the top model features, and the risk segment with its profile. In a real deployment, sending health-related data to an external service would still require consent and compliance with data-protection rules.
- **Fairness and data limits:** the models are based on the CDC BRFSS 2015 survey, which is self-reported data from one country and one year. The results may not apply to other populations or to current conditions, and differences between groups were not evaluated in this project.

### Limitations

- **Variable outputs:** the output changes between runs even at temperature 0.2. Only three test cases were used for each template, so the results are indicative, not conclusive.
- **Global indicators:** the "key contributing indicators" are the top features of the supervised model overall (global feature importance), so every patient receives the same indicators. They are not calculated for each individual patient and do not explain why this specific patient received this probability. Because of this, the generated text can read as if the patient had these conditions, even when the segment profile says otherwise (in our tests, high blood pressure was listed for a Low-risk patient whose profile describes normal blood pressure).
- **Evaluation:** the reviewer is a language model and not a person, and the keyword and readability metrics do not detect wrong statements. No clinicians or real users evaluated the briefings.
- **Readability and language:** the briefings are still written at a college reading level and only in English.
- **External dependency:** the briefing depends on an external API. During testing the service returned a rate-limit error, so availability, usage limits and possible model changes must be considered. The dashboard shows a fixed message when the briefing is unavailable.
- **Scope:** the system is a screening aid built for a course project. It has not been validated clinically and must not be used to make medical decisions.